# DS08 · Bootstrap uncertainty at the right unit

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Data 8: Bootstrap](https://github.com/data-8/textbook/blob/5235b7653f8dfaeb90e43419b9aa069322f2d60b/chapters/13/2/Bootstrap.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

The bootstrap approximates repeated sampling by resampling from the observed dataset. Its validity depends on whether that resampling resembles the way independent information entered the study. For a participant-level estimand with repeated visits, resample participants and preserve all their within-person measurements, or use a justified participant summary. Resampling individual scan rows can destroy the dependence structure.

A bootstrap interval is not a guarantee that the true value has a 95% probability of lying in this particular realized interval under a frequentist interpretation. It is an interval procedure intended to have useful repeated-sampling behavior under assumptions. Percentile intervals can work poorly for small samples, bias, tail quantities, boundaries, or unrepresentative samples. More bootstrap replicates reduce simulation noise, not sampling bias.

The lab compares a person-level bootstrap with an intentionally invalid row bootstrap using a shared random participant effect. Both estimate the same numeric overall mean in this balanced example, but the row bootstrap understates the variability. With unequal visits, the estimands could diverge too. AI should name the resampling unit, dependence preserved, statistic, interval construction, and seed. For a real imaging prediction model, resampling fixed test predictions also leaves out training-set uncertainty; do not call that a full uncertainty analysis of the learning procedure.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Before code, ask me which block must stay together during resampling. Estimate mean uncertainty using participant means and compare an invalid independent-row bootstrap. Explain which uncertainty is and is not included.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Use balanced repeated data so the mean agrees but uncertainty differs. Resample 2,000 times with a fixed seed; compare percentile widths. Deliberate error: increase replicate count and claim it repaired biased participant sampling.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import numpy as np
rng=np.random.default_rng(308)
visits=rng.normal(0,2,(35,1))+rng.normal(0,.2,(35,8))
means=visits.mean(axis=1); flat=visits.ravel()
boot_people=rng.choice(means,(2000,len(means)),replace=True).mean(axis=1)
boot_rows=rng.choice(flat,(2000,len(flat)),replace=True).mean(axis=1)
ci_people=np.quantile(boot_people,[.025,.975]); ci_rows=np.quantile(boot_rows,[.025,.975])
assert np.ptp(ci_people) > 1.8*np.ptp(ci_rows)
print('Participant CI:',ci_people,'invalid row CI:',ci_rows)
print('Bootstrap MC replication count:',len(boot_people))

Participant CI: [-0.83757168  0.33441125] invalid row CI: [-0.4663824  -0.04335328]
Bootstrap MC replication count: 2000


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

The participant interval should be much wider in this construction. More replicates stabilize the simulated endpoints; they do not add participants or fix a nonrepresentative sample.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.